# Log Prediction - .XTF sonar logs to geotagged detections

Runs the trained model on raw **`.xtf` sonar log files** placed in `input/logs/xtf_files/`.

Pipeline per log:
1. **Extract** - `pyxtf` parses the XTF, pings are stacked per channel (port/starboard) and
   rendered to 8-bit waterfall images (dB + percentile stretch).
2. **Save images** - full waterfall strips and 1024x1024 tiles go to `input/log_images/`.
3. **Noise filter** - every tile is cleaned with the same `preprocess_for_model()` from
   `noise_filtering.ipynb` and copied to `output/noise_filter/log_images/`.
4. **Predict** - YOLO detects objects per tile; detections repeated across tile seams are
   merged (IoU + row-closeness).
5. **Detected images** - annotated tiles (and annotated strip overviews) are saved to
   `output/log_prediction/images/`.
6. **Geotag** - each detection is mapped to a ping row, lat/lon/time is interpolated from the
   XTF navigation packets and written to `output/log_prediction/geotag/detections.csv` +
   `detections.geojson`.
7. **Video** - an MP4 review of the survey is written to `output/log_prediction/video/`.

```
input/
  logs/xtf_files/            <- drop .xtf log files here
  log_images/<survey>/       <- extracted waterfall strips + tiles + manifest + nav
output/
  noise_filter/log_images/   <- noise-filtered (model input) copies of the tiles
  log_prediction/
    images/                  <- DETECTED images (annotated tiles + strips)
    geotag/                  <- detections.csv + detections.geojson
    video/                   <- optional MP4 review
```

## 0. Dependencies

"Run All" auto-installs anything that is missing (`pyxtf`, `ultralytics`).

In [ ]:
import subprocess, sys, importlib

for _pkg in ("ultralytics", "pyxtf"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

print("Dependencies ready.")


## 1. Setup

In [ ]:
import json
import math
import subprocess
import sys
import importlib
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    ROOT = Path("/content/SonarVision")
else:
    ROOT = Path.cwd()
    while ROOT.name != "SonarVision" and ROOT.parent != ROOT:
        ROOT = ROOT.parent
    if ROOT.parent == ROOT:
        ROOT = Path(r"D:\1. Project Program\1.SIH\SonarVision")
print("ROOT:", ROOT)

sys.path.insert(0, str(ROOT / "backend"))
import sonar_ingest
from sonar_ingest import extract_survey, render_waterfall, make_tiles, merge_detections


def load_noise_filter_from_nb(nb_path):
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    ns = {"__name__": "noise_filtering_mod", "cv2": cv2, "np": np, "Path": Path}
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if "def filter_noise" in source or "def preprocess_for_model" in source:
            exec(source, ns)
    return ns


NF_NB = ROOT / "backend" / "noise_filtering.ipynb"
if not NF_NB.exists():
    NF_NB = ROOT / "noise_filtering.ipynb"
fns = load_noise_filter_from_nb(NF_NB)
preprocess_for_model = fns["preprocess_for_model"]

BEST = None
for _cand in (ROOT / "backend" / "best" / "best.pt", ROOT / "backend" / "best.pt"):
    if _cand.exists():
        BEST = _cand
        break

model = None
CLASS_NAMES = {}
if BEST is not None:
    model = YOLO(str(BEST))
    CLASS_NAMES = model.names
    print("Model loaded:", BEST)
else:
    print("WARNING: best.pt not found - only extraction + noise filtering will run.")


## 2. Configuration

Paths follow the requested layout:

| Purpose | Folder |
|---|---|
| input logs  | `input/logs/xtf_files/` (recursive scan for `*.xtf`) |
| extracted images | `input/log_images/<survey>/` (strips + tiles) |
| noise-filtered tiles | `output/noise_filter/log_images/` |
| detected images | `output/log_prediction/images/` |
| geotag csv/geojson | `output/log_prediction/geotag/` |
| review video | `output/log_prediction/video/` |

In [ ]:
XTF_DIR      = ROOT / "input" / "logs" / "xtf_files"          # *.xtf sources
IMG_OUT      = ROOT / "input" / "log_images"                  # extracted strips + tiles
FILTERED_OUT = ROOT / "output" / "noise_filter" / "log_images"  # filtered tiles
PRED_OUT     = ROOT / "output" / "log_prediction"             # predictions root
IMAGES_OUT   = PRED_OUT / "images"                            # annotated detected images
GEOTAG_OUT   = PRED_OUT / "geotag"                            # detections.csv / .geojson
VIDEO_OUT    = PRED_OUT / "video"                             # optional MP4 review

# Image generation
STRETCH_PERCENTILE = 1.0       # percentiles clipped at each end (16-bit -> 8-bit)
USE_DB             = True      # apply 20*log10 to intensities before stretching
TILE_SIZE          = 1024      # model input size (matches preprocess_for_model default)
TILE_OVERLAP       = 128       # row overlap between consecutive tiles

# Prediction
CONF_THRESHOLD   = 0.25        # min confidence to keep a detection
IOU_MERGE        = 0.35        # IoU for merging detections that straddle tile seams
MAX_VIDEO_FRAMES = 1500        # skip the MP4 review when a survey is longer than this
MAKE_VIDEO       = True        # generate the MP4 review

for _d in (XTF_DIR, IMG_OUT, FILTERED_OUT, IMAGES_OUT, GEOTAG_OUT, VIDEO_OUT):
    _d.mkdir(parents=True, exist_ok=True)
print("Folders ready.")


## 3. Stage A - Extract images from `.xtf`

Scans `input/logs/xtf_files/` (recursively) for `*.xtf` logs. For each log it:

- reads the sonar channels (port / starboard) with `pyxtf`,
- stacks pings into waterfall strips and renders them to 8-bit,
- saves **full strips** and **1024x1024 tiles** into `input/log_images/<survey>/`,
- writes `manifest.json` (tile -> ping-row mapping) and `nav.npz` (per-ping
  time / lat / lon) used later for geotagging.

> Large logs are read fully by `pyxtf`; for multi-GB files expect a high memory
> footprint. Set `USE_DB = False` for a faster (linear) render if needed.

In [ ]:
def run_extraction(files):
    surveys = []
    for f in files:
        print("Extracting:", f.name)
        survey = extract_survey(f)
        survey_dir = IMG_OUT / survey["name"]
        tiles_dir = survey_dir / "tiles"
        survey_dir.mkdir(parents=True, exist_ok=True)
        tiles_dir.mkdir(parents=True, exist_ok=True)

        manifest = {
            "survey": survey["name"],
            "source": str(f),
            "channels": [],
            "tiles": [],
        }

        for ch in survey["channels"]:
            ch_info = survey["channels"][ch]
            img = render_waterfall(
                ch_info["array"],
                use_db=USE_DB,
                stretch_percentile=STRETCH_PERCENTILE,
            )

            # Full waterfall strip (downscaled preview so it stays small)
            strip_path = survey_dir / "{}_waterfall.png".format(ch)
            disp = img
            if img.shape[0] > 3000:
                scale = 3000.0 / img.shape[0]
                disp = cv2.resize(
                    img,
                    (max(int(img.shape[1] * scale), 1), 3000),
                    interpolation=cv2.INTER_AREA,
                )
            cv2.imwrite(str(strip_path), disp)

            manifest["channels"].append(
                {
                    "label": ch,
                    "side": ch_info["side"],
                    "freq_hz": ch_info["freq_hz"],
                    "pings": int(ch_info["array"].shape[0]),
                    "samples": int(ch_info["array"].shape[1]),
                }
            )

            # Overlapping tiles for the detector
            for tile in make_tiles(img, TILE_SIZE, TILE_OVERLAP):
                tile_name = "{}_r{:06d}.png".format(ch, tile["row0"])
                cv2.imwrite(str(tiles_dir / tile_name), tile["tile"])
                manifest["tiles"].append(
                    {
                        "tile": tile_name,
                        "channel": ch,
                        "side": ch_info["side"],
                        "row_start": tile["row0"],
                        "row_end": tile["row1"],
                        "cols": int(img.shape[1]),
                    }
                )

        np.savez(
            survey_dir / "nav.npz",
            ping_time=survey["ping_time"],
            lat=survey["lat"],
            lon=survey["lon"],
            grid_is_latlon=int(survey["grid_is_latlon"]),
        )
        (survey_dir / "manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )

        surveys.append(survey)
        print(
            "   {} | channels={} | pings={} | tiles={} | nav_is_latlon={}".format(
                survey["name"],
                len(survey["channels"]),
                len(survey["ping_time"]),
                len(manifest["tiles"]),
                survey["grid_is_latlon"],
            )
        )
    return surveys


xtf_files = sorted(XTF_DIR.rglob("*.xtf"))
if not xtf_files:
    print("No .xtf files found under", XTF_DIR)
else:
    print("Found", len(xtf_files), "log file(s)")

surveys = run_extraction(xtf_files) if xtf_files else []
print("Extracted", len(surveys), "survey(s) ->", IMG_OUT)


## 4. Noise-filter the tiles

Applies the same preprocessing used during training (`preprocess_for_model`):
median -> bilateral -> CLAHE, resized to 1024x1024. Results are written to
`output/noise_filter/log_images/` and are what the detector actually sees.

In [ ]:
def run_filter():
    counts = {}
    for survey_dir in sorted(p for p in IMG_OUT.iterdir() if p.is_dir()):
        tiles_dir = survey_dir / "tiles"
        if not tiles_dir.exists():
            continue
        out_dir = FILTERED_OUT / survey_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)
        n = 0
        for tile in sorted(tiles_dir.glob("*.png")):
            filtered = preprocess_for_model(str(tile))
            cv2.imwrite(str(out_dir / tile.name), filtered)
            n += 1
        counts[survey_dir.name] = n
        print("filtered:", survey_dir.name, "-", n, "tiles")
    return counts


filtered_counts = run_filter()


## 5. Stage B - Predict + annotate

Runs the model on every noise-filtered tile, expresses every box in **survey
(strip) row coordinates** and merges detections that repeated across overlapping
tiles. Annotated tiles and annotated strip overviews are saved to
`output/log_prediction/images/`.

If `best.pt` is missing the previous sections still run and this stage prints a
warning.

In [ ]:
def boxes_in_tile(merged, row0):
    out = []
    for b in merged:
        if b["y2"] >= row0 and b["y1"] < row0 + TILE_SIZE:
            bb = dict(b)
            bb["y1"] = max(b["y1"] - row0, 0)
            bb["y2"] = min(b["y2"] - row0, TILE_SIZE)
            out.append(bb)
    return out


def draw_boxes(img_bgr, boxes, label_map):
    for b in boxes:
        x1, y1 = int(round(b["x1"])), int(round(b["y1"]))
        x2, y2 = int(round(b["x2"])), int(round(b["y2"]))
        x1 = max(x1, 0)
        y1 = max(y1, 0)
        x2 = min(x2, img_bgr.shape[1] - 1)
        y2 = min(y2, img_bgr.shape[0] - 1)
        if x2 <= x1 or y2 <= y1:
            continue
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
        lbl = "{} {:.0f}%".format(label_map.get(b["cls"], b["cls"]), b["score"] * 100)
        cv2.putText(
            img_bgr, lbl, (x1, max(y1 - 6, 0)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
        )
    return img_bgr


def run_prediction():
    dets_by_survey = {}
    if model is None:
        print("WARNING: model is None - prediction skipped.")
        return dets_by_survey

    for survey_dir in sorted(p for p in IMG_OUT.iterdir() if p.is_dir()):
        manifest_path = survey_dir / "manifest.json"
        if not manifest_path.exists():
            continue
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        survey_name = survey_dir.name
        side_map = {c["label"]: c["side"] for c in manifest["channels"]}
        filtered_dir = FILTERED_OUT / survey_name
        annot_dir = IMAGES_OUT / survey_name
        annot_dir.mkdir(parents=True, exist_ok=True)

        dets = []
        for entry in manifest["tiles"]:
            tile_rel = entry["tile"]
            fimg = filtered_dir / tile_rel
            if not fimg.exists():
                continue
            results = model.predict(
                source=str(fimg), imgsz=TILE_SIZE,
                conf=CONF_THRESHOLD, verbose=False,
            )
            boxes = results[0].boxes
            if boxes is None or len(boxes) == 0:
                continue
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            clss = boxes.cls.cpu().numpy().astype(int)
            for bx, cf, ci in zip(xyxy, confs, clss):
                x1, y1, x2, y2 = [float(v) for v in bx]
                dets.append(
                    {
                        "x1": x1,
                        "y1": entry["row_start"] + y1,
                        "x2": x2,
                        "y2": entry["row_start"] + y2,
                        "score": float(cf),
                        "cls": int(ci),
                        "tile": tile_rel,
                        "channel": entry["channel"],
                        "side": side_map.get(entry["channel"], ""),
                        "survey": survey_name,
                    }
                )

        merged = merge_detections(
            dets, IOU_MERGE, max(TILE_OVERLAP * 0.5, 1.0)
        ) if dets else []
        dets_by_survey[survey_name] = merged

        # Annotated tiles
        for entry in manifest["tiles"]:
            tile_rel = entry["tile"]
            fimg = filtered_dir / tile_rel
            if not fimg.exists():
                continue
            img = cv2.imread(str(fimg))
            boxes = boxes_in_tile(merged, entry["row_start"])
            img = draw_boxes(img, boxes, CLASS_NAMES)
            cv2.imwrite(str(annot_dir / tile_rel), img)

        # Annotated strip overviews (one per channel)
        for ch_entry in manifest["channels"]:
            strip_path = survey_dir / "{}_waterfall.png".format(ch_entry["label"])
            if not strip_path.exists():
                continue
            strip_bgr = cv2.cvtColor(
                cv2.imread(str(strip_path), cv2.IMREAD_GRAYSCALE),
                cv2.COLOR_GRAY2BGR,
            )
            ch_boxes = [b for b in merged if b["channel"] == ch_entry["label"]]
            strip_bgr = draw_boxes(strip_bgr, ch_boxes, CLASS_NAMES)
            if strip_bgr.shape[0] > 3000:
                scale = 3000.0 / strip_bgr.shape[0]
                strip_bgr = cv2.resize(
                    strip_bgr,
                    (max(int(strip_bgr.shape[1] * scale), 1), 3000),
                    interpolation=cv2.INTER_AREA,
                )
            cv2.imwrite(
                str(annot_dir / "{}_annotated_waterfall.png".format(ch_entry["label"])),
                strip_bgr,
            )

        print(
            "predicted:", survey_name,
            "- raw boxes:", len(dets), "- merged detections:", len(merged),
        )
    return dets_by_survey


dets_by_survey = run_prediction()
print("Annotated images ->", IMAGES_OUT)


## 6. Geotag detections (CSV + GeoJSON)

Each merged detection is projected to the ping row at its vertical centre, and
lat/lon/time is interpolated from the navigation packets stored in `nav.npz`.
Writes `detections.csv` and `detections.geojson` to `output/log_prediction/geotag/`.

> When the XTF navigation is registered in a local grid (not WGS84 lat/lon), the
> CSV still carries the raw coordinates but the GeoJSON features are skipped.

In [ ]:
def write_geotag(dets_by_survey):
    GEOTAG_OUT.mkdir(parents=True, exist_ok=True)
    csv_path = GEOTAG_OUT / "detections.csv"
    geojson_path = GEOTAG_OUT / "detections.geojson"

    header = [
        "survey", "channel", "side", "class_id", "class_name", "confidence",
        "ping_row", "utc_seconds", "lat", "lon",
        "strip_x1", "strip_y1", "strip_x2", "strip_y2", "tile",
    ]
    features = []
    total = 0
    with open(str(csv_path), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for survey_name in sorted(dets_by_survey):
            nav_path = IMG_OUT / survey_name / "nav.npz"
            if not nav_path.exists():
                continue
            nav = np.load(nav_path)
            n = int(nav["lat"].shape[0])
            grid_is_latlon = bool(nav["grid_is_latlon"].tolist())
            for obj in dets_by_survey[survey_name]:
                r = int(max(0.0, min((obj["y1"] + obj["y2"]) // 2, n - 1)))
                lat = float(nav["lat"][r])
                lon = float(nav["lon"][r])
                t = float(nav["ping_time"][r])
                total += 1
                writer.writerow(
                    [
                        survey_name,
                        obj["channel"],
                        obj["side"],
                        obj["cls"],
                        CLASS_NAMES.get(obj["cls"], str(obj["cls"])),
                        round(obj["score"], 4),
                        r,
                        round(t, 3),
                        (round(lat, 7) if not math.isnan(lat) else ""),
                        (round(lon, 7) if not math.isnan(lon) else ""),
                        int(obj["x1"]), int(obj["y1"]),
                        int(obj["x2"]), int(obj["y2"]),
                        obj["tile"],
                    ]
                )
                if grid_is_latlon and not math.isnan(lat) and not math.isnan(lon):
                    features.append(
                        {
                            "type": "Feature",
                            "geometry": {
                                "type": "Point",
                                "coordinates": [lon, lat],
                            },
                            "properties": {
                                "survey": survey_name,
                                "channel": obj["channel"],
                                "side": obj["side"],
                                "class": CLASS_NAMES.get(obj["cls"], str(obj["cls"])),
                                "class_id": obj["cls"],
                                "confidence": round(obj["score"], 4),
                                "time_utc": t,
                                "ping_row": r,
                            },
                        }
                    )

    geojson = {"type": "FeatureCollection", "features": features}
    geojson_path.write_text(json.dumps(geojson, indent=2), encoding="utf-8")
    return total


import csv as _csv  # noqa: E402  (keep csv import next to usage)
csv = _csv

n_dets = write_geotag(dets_by_survey)
print("detections written:", n_dets)
print("->", GEOTAG_OUT / "detections.csv")
print("->", GEOTAG_OUT / "detections.geojson")


## 7. Video review (optional)

Stitches the annotated tiles of each survey into an MP4 for a "play back the log"
review. Skipped for surveys longer than `MAX_VIDEO_FRAMES` tiles.

In [ ]:
def make_video(dets_by_survey):
    if not MAKE_VIDEO or model is None:
        print("video skipped (MAKE_VIDEO=False or model is None)")
        return
    for survey_name in sorted(dets_by_survey):
        annot_dir = IMAGES_OUT / survey_name
        tiles = sorted(annot_dir.glob("ch*_r*.png"))
        if not tiles:
            continue
        if len(tiles) > MAX_VIDEO_FRAMES:
            print("video skipped (long survey):", survey_name, len(tiles), "tiles")
            continue
        out_video = VIDEO_OUT / (survey_name + ".mp4")
        out_video.parent.mkdir(parents=True, exist_ok=True)
        writer = None
        for tp in tiles:
            frame = cv2.imread(str(tp))
            if writer is None:
                writer = cv2.VideoWriter(
                    str(out_video),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    8.0,
                    (frame.shape[1], frame.shape[0]),
                )
            writer.write(frame)
        if writer is not None:
            writer.release()
            print("video ->", out_video)


make_video(dets_by_survey)


## 8. Summary / smoke test

Quick overview of what was produced, plus inline previews of the extracted
strips (with the smoke-test question in mind: *does the detector fire with
reasonable confidence on real sonar?*).

In [ ]:
print("=== SUMMARY ===")
print("Surveys extracted :", len(surveys) if surveys else 0)
print("Filtered tiles   :", sum(filtered_counts.values()) if filtered_counts else 0)
print("Merged detections:", sum(len(v) for v in dets_by_survey.values()))
print()
for survey_name, dets in dets_by_survey.items():
    print("Survey:", survey_name)
    if dets:
        for d in sorted(dets, key=lambda d: -d["score"])[:10]:
            lbl = CLASS_NAMES.get(d["cls"], d["cls"])
            print(
                "   - {:12s} conf={:.2f} row={:.0f}-{:.0f} tile={}".format(
                    lbl, d["score"], d["y1"], d["y2"], d["tile"]
                )
            )
    else:
        print("   - no detections above conf", CONF_THRESHOLD)
print()

# Inline previews of the extracted waterfalls
shown = 0
for survey_dir in sorted(p for p in IMG_OUT.iterdir() if p.is_dir()):
    strips = sorted(survey_dir.glob("*_waterfall.png"))
    for s in strips[:2]:
        im = cv2.imread(str(s), cv2.IMREAD_GRAYSCALE)
        plt.figure(figsize=(6, 8))
        plt.imshow(im, cmap="gray", aspect="auto")
        plt.title(survey_dir.name + " / " + s.name)
        plt.axis("off")
        plt.show()
    shown += len(strips[:2])
    if shown >= 4:
        break
print("Preview strips shown:", shown)
